# Learning Dynamics {#sec-learning-dynamics}

Here, we describe and implement the learning dynamics of our agents. We start by importing the necessary packages and define some configuation for exporting the this notebook to a python module,

In [1]:
 #| default_exp LearningDynamics

In [2]:
 #| export
import jax
import numpy as np
import jax.numpy as jnp
from jax import jit
from functools import partial
import itertools as it

We also define a small helper function to convert a single number into an one-dimensional array, which is useful for some of the agent implementations later on.

In [3]:
 #| export
def make_variable_vector(variable,  # can be iterable or float or int
                         length:int  # length of the vector
                        ):  # vector
    "Turn a `variable` into a vector or check that `length` is consistent."
    if hasattr(variable, '__iter__'):
        assert len(variable) == length, 'Wrong number given'
        return jnp.array(variable)
    else:
        return jnp.repeat(variable, length)

## Model-based learning dynamics

We start with the model-based learning dynamics, which we implement as a Python class. The two main parameters each learning dynamics object recieves are an environment object `env` (see @sec-uncertain-decision-problem) and the learning rates `alpha` of the agents. The remaining parameters have no effect and exist only that the class below can be used as a drop-in replacement for the more complex learnig dynamics used in the first author's Python package [pyCRLD](https://www.github.com/barfusslab/pyCRLD). The reminder of the `__init__` method ensures that that the environmental dynamics, rewards, and observation are internally coherent.

In [4]:
 #| export
class LearningAgents:
    
    def __init__(self,
                 env,
                 learning_rates,
                 discount_factors=None, # dummy argument for compatibility
                 choice_intensities=None, # dummy argument for compatibility
                 use_prefactor=None # dummy argument for compatibility
                ):
        
        self.env = env
        TransitionTensor = env.T
        RewardTensor = env.R
        ObservationTensor = env.O
    
        R = jnp.array(RewardTensor)
        T = jnp.array(TransitionTensor)
        O = jnp.array(ObservationTensor)
        
        # number of agents
        NrAgt = R.shape[0]  
        assert len(T.shape[1:-1]) == NrAgt, "Inconsistent number of agents"
        assert len(R.shape[2:-1]) == NrAgt, "Inconsistent number of agents"
        assert O.shape[0] == NrAgt, "Inconsistent number of agents"

        # number of actions for each agent        
        NrAct = T.shape[1] 
        assert np.allclose(T.shape[1:-1], NrAct), 'Inconsistent number of actions'
        assert np.allclose(R.shape[2:-1], NrAct), 'Inconsistent number of actions'
        
        # number of states
        NrSts = T.shape[0] 
        assert T.shape[-1] == NrSts, 'Inconsistent number of states'
        assert R.shape[-1] == NrSts, 'Inconsistent number of states'
        assert R.shape[1] == NrSts, 'Inconsistent number of states'
        assert O.shape[1] == NrSts, 'Inconsistent number of states'
        
        # number of observations
        NrObs = O.shape[-1]   

        self.R, self.T, self.O = R, T, O
        self.NrAgt, self.NrAct, self.NrSts, self.NrObs = NrAgt, NrAct, NrSts, NrObs
        self.N, self.M, self.Z, self.Q = NrAgt, NrAct, NrSts, NrObs
        
        self.learning_rates = make_variable_vector(learning_rates, NrAgt)
        
        # state and obs distribution helpers
        self.Omega = self._OtherAgentsActionsSummationTensor()
        self.has_last_statdist = False
        self._last_statedist = jnp.ones(NrSts) / NrSts
        self.has_last_obsdist = False
        self._last_obsdist = jnp.ones((NrAgt, NrObs)) / NrObs

We add a method to the class that returns a random stratygy for each agent,

In [5]:
 #| export
def random_softmax_policy(self:LearningAgents):
    """Softmax policy with random probabilities."""
    expQ = jnp.exp(np.random.randn(self.NrAgt, self.NrObs, self.NrAct))
    return expQ / expQ.sum(axis=-1, keepdims=True)

LearningAgents.random_softmax_policy = random_softmax_policy

and one, where each agent selects each action with equal probability,

In [6]:
 #| export
def zero_intelligence_policy(self:LearningAgents):
    """Policy with equal probabilities."""
    return jnp.ones((self.NrAgt, self.NrObs, self.NrAct)) / float(self.NrAct)
LearningAgents.zero_intelligence_policy = zero_intelligence_policy

### Learning update

The `update` method implements the learning update of the agents' strategy,
$$ X^{i,o^i\!,a^i}_{t+1} = 
\frac{
    X^{i,o^i\!,a^i}_{t} \exp\left(\alpha^i R^{i,o^i\!,a^i}_{X_t} \right)
    }{
    \sum_{b \in \mathcal A^i} X^{i,o^i\!, b}_{t} \exp\left(\alpha^i R^{i,o^i\!, b}_{X_t}\right)
    },
$$.


In [7]:
 #| export
@partial(jit, static_argnums=0)
def update(self: LearningAgents,
           Xioa # Joint strategy
          ) -> tuple:  # (Updated joint strategy, Prediction error)
    """
    Performs a learning update of the joint behavioral rule,
    given joint behavior `Xioa`.
    """
    Rioa = self.Rioa(Xioa)
    n = jnp.newaxis
    XexpaRioa = Xioa * jnp.exp(self.learning_rates[:,n,n] * Rioa)
    Xioa_ = XexpaRioa / XexpaRioa.sum(-1, keepdims=True)
    return Xioa_

LearningAgents.update = update

### Rioa | Average rewards

The `Rioa` method computes the average rewards $R_X^{i, o^i\!, a^i}$ for each agent $i$, observation $o^i$, and action $a^i$. Whenever agent $i$ observes observation $o^i$, the environment is state $s$ with probability $B^{i,o^i\!,s}$. In $s$, all other agents $j\neq i$ behave according to $Y_X^{j,s,a^j}$. The environment transitions to state $\acute s$ with probability $T^{s, a, \acute s}$. And agent $i$ receives reward $R^{i,s,a,\acute s}$. Thus, the behavior-average observation-action reward for action $a^i$ under observation $o^i$ reads,
$$
R_X^{i, o^i\!, a^i} = \sum_{s\in\mathcal S} \sum_{a^j \in \mathcal A^j} \sum_{\acute s \in \mathcal S} \sum_{j\neq i}
B_X^{i,o^i\!, s} Y_X^{j,s,a^j} T^{s,a,\acute s} R^{i,s,a,\acute s}.
$$


In [8]:
 #| export
@partial(jit, static_argnums=0)    
def Rioa(self: LearningAgents, 
        Xioa, 
        Bios=None, 
        Yisa=None):
    """Compute average reward Riosa, given joint policy X """
    # For speed up
    Bios = self.Bios(Xioa) if Bios is None else Bios
    Yisa = self.Yisa(Xioa) if Yisa is None else Yisa
    
    # Variables
    # agent i, act a, state s, next state s_, observation o
    i = 0; a = 1; s = 2; s_ = 3; o = 4
    j2k = list(range(5, 5+self.NrAgt-1))  # other agents
    b2d = list(range(5+self.NrAgt-1, 5+self.NrAgt-1 + self.NrAgt))  # all actions
    e2f = list(range(4+2*self.NrAgt, 4+2*self.NrAgt + self.NrAgt-1))  # all other acts

    sumsis = [[j2k[l], s, e2f[l]] for l in range(self.NrAgt-1)]  # sum inds
    otherY = list(it.chain(*zip((self.NrAgt-1)*[Yisa], sumsis)))
    args = [self.Omega, [i]+j2k+[a]+b2d+e2f, Bios, [i, o, s]] +\
            otherY + [self.T, [s]+b2d+[s_], self.R, [i, s]+b2d+[s_],
            [i, o, a]]

    return jnp.einsum(*args, optimize=True)

LearningAgents.Rioa = Rioa

### Yisa | State strategy

The `Yisa` method computes the state-action strategy $Y_X^{i,s,a^i}$ given the current observation-action policy $X^{i,o^i\!,a^i}$ as, 

$$ Y_X^{j,s,a^j} = \sum_{o^j \in \mathcal O^j} O^{j,s,o^j} X^{j,o^j\!, a^j},$$

In [9]:
 #| export
@partial(jit, static_argnums=0)    
def Yisa(self: LearningAgents, 
         Xioa):
    """
    Compute state-action policy given the current observation-action policy
    """
    i = 0; a = 1; s = 2; o = 4  # variables
    args = [self.O, [i, s, o], Xioa, [i, o, a], [i, s, a]]
    Yisa = jnp.einsum(*args, optimize=True)

    return Yisa         

LearningAgents.Yisa = Yisa  

### Bios | State beliefs given observations

The `Bios` method computes the state beliefs $B_X^{i,o^i\!, s}$ given the current observation-action policy $X^{i,o^i\!,a^i}$ as,

$$B^{i,o^i\!,s}_X = \frac{O^{i,s,o^i} P^s_X}{\sum_s O^{i,s,o^i} P^{s}_X},$$


In [10]:
 #| export
@partial(jit, static_argnums=0)
def Bios(self: LearningAgents,
         Xioa):
    """
    Compute 'belief' that environment is in stats s given agent i
    observes observation o (Bayes Rule)
    
    Unsafe when stationary state distribution is not unique
    (i.e., when policies are too extreme)
    """
    i, s, o = 0, 1, 2 # variables 
    # pS = self.statedist(X) # from full obs base (requires Tss from above)
    pS = self.Ps(Xioa, self._last_statedist)
    self._last_statedist = pS
    self.has_last_statdist = True

    b = jnp.einsum(self.O, [i,s,o], pS, [s], [i,s,o], optimize=True)
    bsum = b.sum(axis=1, keepdims=True)
    bsum = bsum + (bsum == 0)  # to avoid dividing by zero
    Biso = b / bsum
    Bios = jnp.swapaxes(Biso, 1,-1)
    
    return Bios

LearningAgents.Bios = Bios

### Ps | Stationary distribution

The `Ps` method computes the stationary distribution $P^s_X$ over states given the current joint observation-action strategy $X^{i,o^i\!,a^i}$ as left eigenvector of the strategy-average transition matrix $T^{s, \acute s}_X$,

In [11]:
 #| export
@partial(jit, static_argnums=0)  
def Ps(self,
       Xioa,  # Joint strategy
       pS0):  # Last stationary state distribution 
    """
    Compute stationary state distribution `Ps`, given joint strategy `Xioa`
    using JAX.
    """
    Tss = self.Tss(Xioa)
    _pS = compute_stationarydistribution(Tss)
    nrS = jnp.where(_pS.mean(0)!=-10, 1, 0).sum()

    @jit
    def single_dist(pS):
        return jnp.max(jnp.where(_pS.mean(0)!=-10,
                                 jnp.arange(_pS.shape[0]), -1))
    @jit
    def multi_dist(pS):
        ix = jnp.argmin(jnp.linalg.norm(_pS.T - pS0, axis=-1))
        return ix
        
    ix = jax.lax.cond(nrS == 1, single_dist, multi_dist, _pS)

    pS = _pS[:, ix]
    return pS

LearningAgents.Ps = Ps

### Tss | State transition

The `Tss` method computes the strategy-average transition matrix $T^{s, \acute s}_X$ given the current observation-action strategy $X^{i,o^i\!,a^i}$ as,

$$ T^{s, \acute s}_X = \sum_{a^j \in \mathcal A^j} \prod_{j \in \mathcal N}  Y^{j,s,a^j}_X T^{s,a, \acute s},$$

In [12]:
 #| export
@partial(jit, static_argnums=0)
def Tss(self: LearningAgents,
        Xioa):
    """Compute average transition model Tss given policy X"""
    Yisa = self.Yisa(Xioa)

    s = 1  # state s
    sprim = 2  # next state s'
    b2d = list(range(3, 3+self.NrAgt))  # all actions

    X4einsum = list(it.chain(*zip(Yisa, [[s, b2d[a]]
                                         for a in range(self.NrAgt)])))
    args = X4einsum + [self.T, [s]+b2d+[sprim], [s, sprim]]
    return jnp.einsum(*args, optimize=True)

LearningAgents.Tss = Tss

We are left with implementing a few helper functions and methods.

### aux | Stationary distribution

The `compute_stationarydistribution` function computes the stationary distribution as the left eigenvector of the transition matrix with eigenvalue 1.

In [13]:
 #| export
@jit
def compute_stationarydistribution(Tkk:jnp.ndarray):  # Transition matrix
    """Compute stationary distribution for transition matrix `Tkk`."""
    # eigenvectors
    oeival, oeivec = jnp.linalg.eig(Tkk.T)
    oeival = oeival.real
    oeivec = oeivec.real
    
    get_mask = lambda tol: jnp.abs(oeival - 1) < tol
  
    tolerances = jax.lax.map(lambda x: 0.1**x, jnp.arange(1,16,1))
    masks = jax.lax.map(get_mask, tolerances)
    ix = jnp.max(jnp.where(masks.sum(-1)>=1, jnp.arange(len(masks)), -1))
    mask = masks[ix]
    tol = tolerances[ix]
    
    # obtain stationary distribution
    meivec = jnp.where(mask, oeivec, -42)
    
    dist = meivec / meivec.sum(axis=0, keepdims=True)
    dist = jnp.where(dist < tol, 0, dist)
    dist = dist / dist.sum(axis=0, keepdims=True)
    
    return jnp.where(meivec==-42, -10, dist)

### axu | Other agents summation tensor

The `_OtherAgentsActionsSummationTensor` allows to sum over all other agents' actions when computing expected values.

In [14]:
 #| export
def _OtherAgentsActionsSummationTensor(self:LearningAgents):
    """
    To sum over the other agents and their respective actions using `einsum`.
    """
    dim = np.concatenate(([self.NrAgt],  # agent i
                          [self.NrAgt for _ in range(self.NrAgt-1)],  # other agnt
                          [self.NrAct],  # agent a of agent i
                          [self.NrAct for _ in range(self.NrAgt)],  # all acts
                          [self.NrAct for _ in range(self.NrAgt-1)]))  # other a's
    Omega = np.zeros(dim.astype(int), int)

    for index, _ in np.ndenumerate(Omega):
        I = index[0]
        notI = index[1:self.NrAgt]
        A = index[self.NrAgt]
        allA = index[self.NrAgt+1:2*self.NrAgt+1]
        notA = index[2*self.NrAgt+1:]

        if len(np.unique(np.concatenate(([I], notI)))) is self.NrAgt:
            # all agents indices are different

            if A == allA[I]:
                # action of agent i equals some other action
                cd = allA[:I] + allA[I+1:]  # other actionss
                areequal = [cd[k] == notA[k] for k in range(self.NrAgt-1)]
                if np.all(areequal):
                    Omega[index] = 1

    return jnp.array(Omega)
LearningAgents._OtherAgentsActionsSummationTensor = _OtherAgentsActionsSummationTensor

### aux | Trajectory 

The `trajectory` method simulates a learning trajectory from an initial strategy `Xinit` over a maximum of `Tmax` time steps or until convergence within tolerance `tolerance` is reached.

In [15]:
 #| export
def trajectory(self: LearningAgents,
               Xinit:jnp.ndarray,  # Initial condition
               Tmax:int=100, # the maximum number of iteration steps
               tolerance:float=None, # to determine if a fix point is reached 
               verbose=False,  # Say something during computation?
               **kwargs) -> tuple: # (`trajectory`, `fixpointreached`)
    """
    Compute a joint learning trajectory.
    """
    traj = []
    t = 0
    X = Xinit.copy()
    fixpreached = False

    while not fixpreached and t < Tmax:
        print(f"\r [computing trajectory] step {t}", end='') if verbose else None 
        traj.append(X)

        X_ = self.update(X)
        if np.any(np.isnan(X_)):
            fixpreached = True
            break

        if tolerance is not None:
            fixpreached = np.linalg.norm(X_ - X) < tolerance

        X = X_
        t += 1

    print(f" [trajectory computed]") if verbose else None

    return np.array(traj), fixpreached
LearningAgents.trajectory = trajectory

### aux | Average rewards Ri

The `Ri` method computes the average rewards for each agent given the current strategy,

$$ R^i_X = \sum_{o^i \in\mathcal O^i} \sum_{a^i \in \mathcal A^i} 
P^{o^i}_X R^{i,o,a}.
$$

In [16]:
 #| export
# @partial(jit, static_argnums=0)
def Ri(self: LearningAgents, 
       Xioa):
    """Compute average reward Ri, given joint policy X""" 
    i, o, a = 0, 1, 2
    return jnp.einsum(self.obsdist(Xioa), [i, o], 
                      Xioa, [i, o, a],
                      self.Rioa(Xioa), [i, o, a], [i])
LearningAgents.Ri = Ri

To do so, it uses the observation distribution $P^{o^i}_X$.

### aux | Observation distribution

The `obsdist` method computes the observation distribution for each agent given the current strategy as the left eigenvector of their observation-transitions matrix $T^{i,o^i,\acute o^i}$.

In [17]:
 #| export
# @partial(jit, static_argnums=0)
def obsdist(self: LearningAgents, Xioa):
    """Compute stationary distribution, given joint policy X"""
    Tioo = self.Tioo(Xioa)
    Dio = np.zeros((self.NrAgt, self.NrObs))
    
    for i in range(self.NrAgt):
        pO = np.array(compute_stationarydistribution(Tioo[i]))
    
        pO = pO[:, pO.mean(0)!=-10]
        if len(pO[0]) == 0:  # this happens when the tollerance can distin.
            assert False, 'No _statdist return - must not happen'
        elif len(pO[0]) > 1:  # Should not happen, in an ideal world
            # sidenote: This means an ideal world is ergodic ;)
            print("More than 1 state-eigenvector found")
            print(pO.round(2))
            nr = len(pO[0])
            choice = np.random.randint(nr)
            print("taking random one: ", choice)
            pO = pO[:, choice]
                    
        Dio[i] = pO.flatten()

    self._last_obsdist = Dio
    self.has_last_obsdist = True
    return Dio
LearningAgents.obsdist = obsdist

### Tioo | Observation transition matrices

The `Tioo` method computes the observation-transition matrices $T^{i,o^i,\acute o^i}$ for each agent given the current strategy as,

In [18]:
 #| export
@partial(jit, static_argnums=0)    
def Tioo(self, X, Bios=None, Yisa=None):
    """Compute average transition model Tioo, given joint policy X"""
    # For speed up
    Bios = self.Bios(X) if Bios is None else Bios
    Yisa = self.Yisa(X) if Yisa is None else Yisa
    
    # variables 
    # agent i, state s, next state s_, observation o, next obs o', all acts
    i = 0; s = 1; s_ = 2; o = 3; o_ = 4; b2d = list(range(5, 5+self.NrAgt)) 

    Y4einsum = list(it.chain(*zip(Yisa, 
                                  [[s, b2d[a]] for a in range(self.NrAgt)])))
    
    args = [Bios, [i, o, s]] + Y4einsum + [self.T, [s]+b2d+[s_],
            self.O, [i, s_, o_], [i, o, o_]]
    return jnp.einsum(*args, optimize=True)
LearningAgents.Tioo = Tioo

### aux | Identifier

The `id` method returns a string identifier for the learning dynamics object.

In [19]:
 #| export
def id(self: LearningAgents
       ) -> str:  # id
    """Returns an identifier to handle simulation runs."""
    envid = self.env.id() + "__"
    agentsid = f"j{self.__class__.__name__}_"

    agentsid += 'PartObs_'        

    agentsid += f"{str(self.learning_rates)}"
    
    return envid + agentsid
LearningAgents.id = id

## Model-free learning dynamics

Having implemented the model-based learning dynamics, we can now easily implement model-free learning dynamics by inheriting from the model-based class and overriding only the `update` method.

In [20]:
 #| export
class ModelFreeAgents(LearningAgents):
    
     # @partial(jit, static_argnums=0)
     def update(self,
                Xioa # Joint strategy
               ) -> tuple:  # (Updated joint strategy, Prediction error)
          """
          Performs a learning update of the joint behavioral rule,
          given joint behavior `Xioa`.
          """
          Rioa = self.Rioa(Xioa)
          Pio = self.obsdist(Xioa)

          n = jnp.newaxis
          XexpaRioa = Xioa * jnp.exp(self.learning_rates[:,n,n]
                                     * Pio[:,:,n]
                                     * Xioa
                                     * Rioa)
          Xioa_ = XexpaRioa / XexpaRioa.sum(-1, keepdims=True)
          return Xioa_

          LearningAgents.update = update

## Testing the implementation

We test our implementation of the learning dynamics classes on the uncertain decision problem defined in @sec-uncertain-decision-problem.

In [21]:
np.random.seed(42)

In [22]:
from _code.UncertainDecisionProblem import \
    SingleAgentUncertainDecisionProblem, TwoAgentUncertainDecisionProblem

envparams = dict(probabilityA=0.5, likelydistance=0.5)
envclass = TwoAgentUncertainDecisionProblem
envclassS = SingleAgentUncertainDecisionProblem
env = envclass(noiselevel=2.5, **envparams)
envS = envclassS(noiselevel=2.5, **envparams)

print(env.T.shape)
print(env.R.shape)
print(env.O.shape)

(12, 2, 2, 12)
(2, 12, 2, 2, 12)
(2, 12, 13)


### Testing the model-based leaning dynamics

In [23]:
MBA_LA = LearningAgents(env, learning_rates=0.25)
print(MBA_LA.id())
MBA_LAS = LearningAgents(envS, learning_rates=0.25)
print(MBA_LAS.id())

TwoAgentUncertainDecisionProblem_2.5_0.5_0.5__jLearningAgents_PartObs_[0.25 0.25]
SingleAgentUncertainDecisionProblem_2.5_0.5_0.5__jLearningAgents_PartObs_[0.25]


In [24]:
for LA in [MBA_LA, MBA_LAS]:
    Xioa = LA.random_softmax_policy()
    Xtioa, fpr = LA.trajectory(Xioa, Tmax=5000, tolerance=1e-6)
    print(Xtioa.shape, fpr)

(306, 2, 13, 2) True
(523, 1, 4, 2) True


### Testing the model-free leaning dynamics

In [25]:
MFA_LA = ModelFreeAgents(env, learning_rates=0.25)
print(MFA_LA.id())
MFA_LAS = ModelFreeAgents(envS, learning_rates=0.25)
print(MFA_LAS.id())

TwoAgentUncertainDecisionProblem_2.5_0.5_0.5__jModelFreeAgents_PartObs_[0.25 0.25]
SingleAgentUncertainDecisionProblem_2.5_0.5_0.5__jModelFreeAgents_PartObs_[0.25]


In [26]:
for LA in [MFA_LA, MFA_LAS]:
    Xioa = LA.random_softmax_policy()
    Xtioa, fpr = LA.trajectory(Xioa, Tmax=500000, tolerance=1e-6)
    print(Xtioa.shape, fpr)

(10696, 2, 13, 2) True


(5893, 1, 4, 2) True


Finally, we export the learning dynamics classes into their own module,

In [27]:
import nbdev
nbdev.export.nb_export("j02_AGEN_LearningDynamics.ipynb", "_code")